In [2]:
# =============================================================================
# NLP Essentials — Clean Pipeline for NZT text tables (Dissertation-ready)
# Paths aligned with your notebook:
#   OUTDIR = outputs/nzt_gap_overlap/
#   INV_TEXT_ALL = nzt_investor_text_table.csv
#   INV_TEXT_NET = nzt_investor_text_table_investors_only.csv
#   TAILORED_JSON = outputs/nzt_gap_overlap/nzt_tailored_keywords.json (optional)
# =============================================================================

from pathlib import Path
import os, re, json, unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# -----------------------------
# Paths (your originals)
# -----------------------------
OUTDIR        = Path("outputs/nzt_gap_overlap")
OUTDIR.mkdir(parents=True, exist_ok=True)
NZT_XLSX      = Path("Net Zero Tracker_May 2025.xlsx")  # not used here, kept for consistency
NETWORK_FILE  = Path("../WRDS/financed_emissions_network_final_plus_manual.csv")  # not used here

INV_TEXT_ALL  = Path("nzt_investor_text_table.csv")
INV_TEXT_NET  = Path("nzt_investor_text_table_investors_only.csv")
TAILORED_JSON = OUTDIR / "nzt_tailored_keywords.json"   # optional extension of rules

# -----------------------------
# Load table (prefer ALL, fallback to NET)
# -----------------------------
def _load_text_table() -> pd.DataFrame:
    if INV_TEXT_ALL.exists():
        return pd.read_csv(INV_TEXT_ALL)
    elif INV_TEXT_NET.exists():
        return pd.read_csv(INV_TEXT_NET)
    else:
        # Helpful message with contents of cwd
        files = "\n  - " + "\n  - ".join(sorted([p.name for p in Path(".").glob("*")]))
        raise FileNotFoundError(
            "Could not find input CSV. Expected one of:\n"
            f"  • {INV_TEXT_ALL}\n  • {INV_TEXT_NET}\n"
            f"Current directory contains:{files}"
        )

df_raw = _load_text_table()

# -----------------------------
# Column inference (robust to naming)
# -----------------------------
def _norm(s: str) -> str:
    s = s.lower().strip()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9_]+", "_", s)
    return s.strip("_")

cols_map = {c: _norm(c) for c in df_raw.columns}

# Candidate patterns
INV_KEYS  = ("investor", "manager", "owner", "fund", "holder")
ISS_KEYS  = ("issuer", "company", "stock", "security", "firm", "constituent")
TEXT_KEYS = ("coverage_notes", "notes", "text", "statement", "pledge", "description", "disclosure")

def _pick(colmap, candidates):
    for key in candidates:
        for c, n in colmap.items():
            if key in n:
                return c
    return None

COL_INVESTOR = _pick(cols_map, INV_KEYS)
COL_ISSUER   = _pick(cols_map, ISS_KEYS)
COL_TEXT     = _pick(cols_map, TEXT_KEYS)

if COL_INVESTOR is None:
    # fall back to a 'name' like column if present
    COL_INVESTOR = _pick(cols_map, ("name", "entity", "organisation", "organization"))
if COL_TEXT is None:
    raise ValueError(
        "Could not locate a text column. Looked for one of: "
        + ", ".join(TEXT_KEYS) +
        ". Please rename your text field or adjust TEXT_KEYS."
    )

# Keep minimal tidy columns
keep_cols = [COL_INVESTOR, COL_TEXT] + ([COL_ISSUER] if COL_ISSUER else [])
df = df_raw[keep_cols].copy()

# -----------------------------
# Text normalization
# -----------------------------
def normalize_text(s: str) -> str:
    """Lowercase, de-accent, keep %/&/ slash, collapse whitespace."""
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = s.strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^\w\s%&/]", " ", s)   # retain %, &, /
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_raw"]  = df[COL_TEXT].fillna("")
df["text_norm"] = df["text_raw"].map(normalize_text)

# -----------------------------
# Deterministic rules (base set)
# Optionally extended by TAILORED_JSON if present:
#   {"mentions_fossil_fuel": {"keywords": [...], "regex": [...]}, ...}
# -----------------------------
CATEGORY_RULES = {
    "mentions_fossil_fuel": {
        "keywords": [
            "fossil fuel", "fossil-fuel", "coal", "thermal coal", "oil sands",
            "upstream oil", "gas extraction", "oil & gas", "o&g"
        ],
        "regex": [
            r"\bfossil[\s\-]?fuel(s)?\b",
            r"\bthermal\s+coal\b",
            r"\boil\s+sands?\b",
            r"\boil\s*&\s*gas\b|\bo&g\b",
            r"\b(upstream|midstream|downstream)\s+oil\s*&?\s*gas\b"
        ]
    },
    "mentions_67pct_rule": {
        "keywords": ["67%", "sixty-seven percent", "67 percent", "two-thirds"],
        "regex": [
            r"\b67\s?%\b",
            r"\bsixty[\-\s]?seven\s+percent\b",
            r"\b67\s+percent\b",
            r"\btwo[\-\s]?thirds\b"
        ]
    },
    "mentions_scope3_cat15": {
        "keywords": ["scope 3 category 15", "category 15 investments", "financed emissions"],
        "regex": [
            r"\bscope\s*3\s*(category|cat)\s*15\b",
            r"\bcategory\s*15\b",
            r"\bfinanced\s+emissions\b"
        ]
    },
    "mentions_sBTi": {
        "keywords": ["sbti", "science based target initiative", "science-based targets"],
        "regex": [
            r"\bsbti\b",
            r"\bscience[\-\s]based\s+target(s)?(\s+initiative)?\b"
        ]
    },
    "mentions_net_zero": {
        "keywords": ["net zero", "net-zero", "netzero"],
        "regex": [r"\bnet[\-\s]?zero\b"]
    },
}

# Extend with tailored JSON if available
if TAILORED_JSON.exists():
    try:
        user_rules = json.loads(TAILORED_JSON.read_text(encoding="utf-8"))
        for k, v in user_rules.items():
            base = CATEGORY_RULES.get(k, {"keywords": [], "regex": []})
            base["keywords"] = list(dict.fromkeys(list(base.get("keywords", [])) + list(v.get("keywords", []))))
            base["regex"]    = list(dict.fromkeys(list(base.get("regex", []))    + list(v.get("regex", []))))
            CATEGORY_RULES[k] = base
    except Exception as e:
        print(f"[WARN] Could not read tailored rules: {e}")

def find_triggers(text_norm: str, rule_block: dict):
    """Return sorted list of matched keywords/regex for a rule block."""
    hits = set()
    # keyword hits
    for kw in rule_block.get("keywords", []):
        kw_norm = normalize_text(kw)
        if kw_norm and kw_norm in text_norm:
            hits.add(kw.strip())
    # regex hits
    for pat in rule_block.get("regex", []):
        for m in re.finditer(pat, text_norm, flags=re.IGNORECASE):
            frag = m.group(0).strip()
            if frag:
                hits.add(frag)
    return sorted(hits)

def apply_category_rules(df_in: pd.DataFrame, text_col: str) -> pd.DataFrame:
    out = df_in.copy()
    tnorm = out[text_col].fillna("").map(normalize_text)
    for cat, block in CATEGORY_RULES.items():
        terms = tnorm.map(lambda s: find_triggers(s, block))
        out[f"{cat}_flag"]  = terms.map(bool)
        out[f"{cat}_terms"] = terms
    return out

df_tagged = apply_category_rules(df, "text_norm")

# -----------------------------
# TF-IDF soft matching to themes
# -----------------------------
SEED_THEMES = {
    "fossil_fuels" : [
        "fossil fuel phase down", "coal oil gas exclusion",
        "no new coal", "gas plant expansion restriction",
        "thermal coal exit", "oil sands exclusion", "upstream oil & gas"
    ],
    "portfolio_coverage" : [
        "67% of financed emissions", "portfolio coverage target",
        "percentage of financed emissions covered by targets",
        "share of scope 3 category 15 covered"
    ],
    "net_zero_targets" : [
        "net zero target", "science based target initiative",
        "near term target", "long term net zero", "scope 3 category 15"
    ],
}

def tfidf_similarity_scores(texts: pd.Series, seeds: dict) -> pd.DataFrame:
    docs = texts.fillna("").astype(str).tolist()
    theme_names = list(seeds.keys())
    seed_docs = [" ".join(v) for v in seeds.values()]
    corpus = docs + seed_docs

    vect = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        max_df=0.9, min_df=2
    )
    X = vect.fit_transform(corpus)
    D, S = X[:len(docs)], X[len(docs):]
    sims = cosine_similarity(D, S)
    return pd.DataFrame(sims, columns=[f"sim_tfidf__{t}" for t in theme_names])

tfidf_scores = tfidf_similarity_scores(df_tagged["text_raw"], SEED_THEMES)

# -----------------------------
# Optional sentence embeddings (auto-skip)
# -----------------------------
def embedding_similarity_scores(texts: pd.Series, seeds: dict) -> pd.DataFrame:
    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.metrics.pairwise import cosine_similarity as cs
    except Exception:
        return pd.DataFrame(index=range(len(texts)))

    model_name = "all-MiniLM-L6-v2"
    try:
        model = SentenceTransformer(model_name)
    except Exception:
        return pd.DataFrame(index=range(len(texts)))

    docs      = texts.fillna("").astype(str).tolist()
    themes    = list(seeds.keys())
    seed_docs = [" ".join(v) for v in seeds.values()]

    E_docs  = model.encode(docs,  show_progress_bar=False, normalize_embeddings=True)
    E_seeds = model.encode(seed_docs, show_progress_bar=False, normalize_embeddings=True)
    sims = cs(E_docs, E_seeds)
    return pd.DataFrame(sims, columns=[f"sim_embed__{t}" for t in themes])

embed_scores = embedding_similarity_scores(df_tagged["text_raw"], SEED_THEMES)

# -----------------------------
# Merge & export
# -----------------------------
out = pd.concat([df_tagged.reset_index(drop=True), tfidf_scores, embed_scores], axis=1)

# Investor/Issuer/Text column names for tidy export
export_cols = []
for c in [COL_INVESTOR, COL_ISSUER, COL_TEXT]:
    if c and c in df.columns and c not in export_cols:
        export_cols.append(c)
export_cols += [c for c in out.columns if c not in export_cols]  # append derived metrics

# Save main results
out_path = OUTDIR / "nlp_essentials_results.csv"
out[export_cols].to_csv(out_path, index=False)

# Audits requested: fossil-fuel and 67% mentions by investor
inv_col = COL_INVESTOR if COL_INVESTOR else export_cols[0]
audit_cols = [inv_col] + ([COL_ISSUER] if COL_ISSUER else []) + [
    "mentions_fossil_fuel_flag", "mentions_fossil_fuel_terms",
    "mentions_67pct_rule_flag",  "mentions_67pct_rule_terms"
]

audit_fossil = out[out["mentions_fossil_fuel_flag"]][audit_cols].sort_values(inv_col)
audit_67pct  = out[out["mentions_67pct_rule_flag"]][audit_cols].sort_values(inv_col)

audit_fossil_path = OUTDIR / "audit_investors_fossil_fuel.csv"
audit_67pct_path  = OUTDIR / "audit_investors_67pct.csv"
audit_fossil.to_csv(audit_fossil_path, index=False)
audit_67pct.to_csv(audit_67pct_path, index=False)

print(f"[OK] Results: {out_path}")
print(f"[OK] Audit (fossil-fuel): {audit_fossil_path}")
print(f"[OK] Audit (67%): {audit_67pct_path}")

# =============================================================================
# OPTIONAL EXTRAS (COMMENTED — keep pipeline lean for dissertation)
# - Threshold TF-IDF/embedding scores for additional binary flags
# - Small validation set for precision/recall
# - Plots of similarity distributions / error analysis near thresholds
# =============================================================================

# Example thresholding (commented):
# THR = {"fossil_fuels": 0.10, "portfolio_coverage": 0.12, "net_zero_targets": 0.10}
# for theme, thr in THR.items():
#     col = f"sim_tfidf__{theme}"
#     if col in out.columns:
#         out[f"flag_tfidf__{theme}"] = out[col] >= thr
# (OUTDIR / "nlp_essentials_results_thresholded.csv").write_text(
#     out.to_csv(index=False)
# )


[WARN] Could not read tailored rules: 'list' object has no attribute 'get'
[OK] Results: outputs/nzt_gap_overlap/nlp_essentials_results.csv
[OK] Audit (fossil-fuel): outputs/nzt_gap_overlap/audit_investors_fossil_fuel.csv
[OK] Audit (67%): outputs/nzt_gap_overlap/audit_investors_67pct.csv


In [3]:
# =============================================================================
# NLP Essentials — Display results in notebook (no files)
# Aligned with your paths; robust to list/dict tailored JSON
# =============================================================================

from pathlib import Path
import os, re, json, unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML

# Pretty display
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
SHOW_N = 50  # how many rows to show per table

# -----------------------------
# Paths (yours)
# -----------------------------
OUTDIR        = Path("outputs/nzt_gap_overlap")
OUTDIR.mkdir(parents=True, exist_ok=True)
NZT_XLSX      = Path("Net Zero Tracker_May 2025.xlsx")  # not used here
NETWORK_FILE  = Path("../WRDS/financed_emissions_network_final_plus_manual.csv")  # not used here

INV_TEXT_ALL  = Path("nzt_investor_text_table.csv")
INV_TEXT_NET  = Path("nzt_investor_text_table_investors_only.csv")
TAILORED_JSON = OUTDIR / "nzt_tailored_keywords.json"   # optional

# -----------------------------
# Load table (prefer ALL, fallback to NET)
# -----------------------------
def _load_text_table() -> pd.DataFrame:
    if INV_TEXT_ALL.exists():
        return pd.read_csv(INV_TEXT_ALL)
    elif INV_TEXT_NET.exists():
        return pd.read_csv(INV_TEXT_NET)
    else:
        files = "\n  - " + "\n  - ".join(sorted([p.name for p in Path('.').glob('*')]))
        raise FileNotFoundError(
            "Could not find input CSV. Expected one of:\n"
            f"  • {INV_TEXT_ALL}\n  • {INV_TEXT_NET}\n"
            f"Current directory contains:{files}"
        )

df_raw = _load_text_table()

# -----------------------------
# Column inference (robust to naming)
# -----------------------------
def _norm(s: str) -> str:
    s = s.lower().strip()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^a-z0-9_]+", "_", s)
    return s.strip("_")

cols_map = {c: _norm(c) for c in df_raw.columns}

INV_KEYS  = ("investor", "manager", "owner", "fund", "holder")
ISS_KEYS  = ("issuer", "company", "stock", "security", "firm", "constituent")
TEXT_KEYS = ("coverage_notes", "notes", "text", "statement", "pledge", "description", "disclosure", "comment")

def _pick(colmap, candidates):
    for key in candidates:
        for c, n in colmap.items():
            if key in n:
                return c
    return None

COL_INVESTOR = _pick(cols_map, INV_KEYS) or _pick(cols_map, ("name","entity","organisation","organization"))
COL_ISSUER   = _pick(cols_map, ISS_KEYS)
COL_TEXT     = _pick(cols_map, TEXT_KEYS)

if COL_TEXT is None:
    raise ValueError("Could not locate a text column. Consider renaming one to include any of: " + ", ".join(TEXT_KEYS))

keep_cols = [COL_INVESTOR, COL_TEXT] + ([COL_ISSUER] if COL_ISSUER else [])
df = df_raw[keep_cols].copy()

# -----------------------------
# Text normalization
# -----------------------------
def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = s.strip().lower()
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r"[^\w\s%&/]", " ", s)   # keep %, &, /
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_raw"]  = df[COL_TEXT].fillna("")
df["text_norm"] = df["text_raw"].map(normalize_text)

# -----------------------------
# Deterministic rules (base set)
# + robust merge with tailored JSON (dict OR list)
# -----------------------------
CATEGORY_RULES = {
    "mentions_fossil_fuel": {
        "keywords": [
            "fossil fuel", "fossil-fuel", "coal", "thermal coal", "oil sands",
            "upstream oil", "gas extraction", "oil & gas", "o&g"
        ],
        "regex": [
            r"\bfossil[\s\-]?fuel(s)?\b",
            r"\bthermal\s+coal\b",
            r"\boil\s+sands?\b",
            r"\boil\s*&\s*gas\b|\bo&g\b",
            r"\b(upstream|midstream|downstream)\s+oil\s*&?\s*gas\b"
        ]
    },
    "mentions_67pct_rule": {
        "keywords": ["67%", "sixty-seven percent", "67 percent", "two-thirds"],
        "regex": [
            r"\b67\s?%\b",
            r"\bsixty[\-\s]?seven\s+percent\b",
            r"\b67\s+percent\b",
            r"\btwo[\-\s]?thirds\b"
        ]
    },
    "mentions_scope3_cat15": {
        "keywords": ["scope 3 category 15", "category 15 investments", "financed emissions"],
        "regex": [
            r"\bscope\s*3\s*(category|cat)\s*15\b",
            r"\bcategory\s*15\b",
            r"\bfinanced\s+emissions\b"
        ]
    },
    "mentions_sBTi": {
        "keywords": ["sbti", "science based target initiative", "science-based targets"],
        "regex": [
            r"\bsbti\b",
            r"\bscience[\-\s]based\s+target(s)?(\s+initiative)?\b"
        ]
    },
    "mentions_net_zero": {
        "keywords": ["net zero", "net-zero", "netzero"],
        "regex": [r"\bnet[\-\s]?zero\b"]
    },
}

def _merge_rule_into(base: dict, rule_name: str, rule_obj: dict):
    base_entry = base.get(rule_name, {"keywords": [], "regex": []})
    kws = list(base_entry.get("keywords", [])) + list(rule_obj.get("keywords", []))
    rxs = list(base_entry.get("regex", []))    + list(rule_obj.get("regex", []))
    # dedupe while preserving order
    base[rule_name] = {
        "keywords": list(dict.fromkeys(kws)),
        "regex":    list(dict.fromkeys(rxs))
    }

# Try to extend with tailored JSON (supports dict or list-of-dicts)
if TAILORED_JSON.exists():
    try:
        data = json.loads(TAILORED_JSON.read_text(encoding="utf-8"))
        if isinstance(data, dict):
            for name, rule in data.items():
                if isinstance(rule, dict):
                    _merge_rule_into(CATEGORY_RULES, name, rule)
        elif isinstance(data, list):
            for item in data:
                if isinstance(item, dict):
                    # require a "name" field to identify rule
                    name = item.get("name")
                    if name:
                        _merge_rule_into(CATEGORY_RULES, name, item)
        else:
            print("[WARN] Tailored rules JSON must be a dict or a list of dicts.")
    except Exception as e:
        print(f"[WARN] Could not read tailored rules: {e}")

def find_triggers(text_norm: str, rule_block: dict):
    hits = set()
    for kw in rule_block.get("keywords", []):
        kw_norm = normalize_text(kw)
        if kw_norm and kw_norm in text_norm:
            hits.add(kw.strip())
    for pat in rule_block.get("regex", []):
        for m in re.finditer(pat, text_norm, flags=re.IGNORECASE):
            frag = m.group(0).strip()
            if frag:
                hits.add(frag)
    return sorted(hits)

def apply_category_rules(df_in: pd.DataFrame, text_col: str) -> pd.DataFrame:
    out = df_in.copy()
    tnorm = out[text_col].fillna("").map(normalize_text)
    for cat, block in CATEGORY_RULES.items():
        terms = tnorm.map(lambda s: find_triggers(s, block))
        out[f"{cat}_flag"]  = terms.map(bool)
        out[f"{cat}_terms"] = terms
    return out

df_tagged = apply_category_rules(df, "text_norm")

# -----------------------------
# TF-IDF soft matching to themes
# -----------------------------
SEED_THEMES = {
    "fossil_fuels" : [
        "fossil fuel phase down", "coal oil gas exclusion",
        "no new coal", "gas plant expansion restriction",
        "thermal coal exit", "oil sands exclusion", "upstream oil & gas"
    ],
    "portfolio_coverage" : [
        "67% of financed emissions", "portfolio coverage target",
        "percentage of financed emissions covered by targets",
        "share of scope 3 category 15 covered"
    ],
    "net_zero_targets" : [
        "net zero target", "science based target initiative",
        "near term target", "long term net zero", "scope 3 category 15"
    ],
}

def tfidf_similarity_scores(texts: pd.Series, seeds: dict) -> pd.DataFrame:
    docs = texts.fillna("").astype(str).tolist()
    theme_names = list(seeds.keys())
    seed_docs = [" ".join(v) for v in seeds.values()]
    corpus = docs + seed_docs

    vect = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        max_df=0.9, min_df=2
    )
    X = vect.fit_transform(corpus)
    D, S = X[:len(docs)], X[len(docs):]
    sims = cosine_similarity(D, S)
    return pd.DataFrame(sims, columns=[f"sim_tfidf__{t}" for t in theme_names])

tfidf_scores = tfidf_similarity_scores(df_tagged["text_raw"], SEED_THEMES)

# -----------------------------
# Optional sentence embeddings (auto-skip if not available)
# -----------------------------
def embedding_similarity_scores(texts: pd.Series, seeds: dict) -> pd.DataFrame:
    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.metrics.pairwise import cosine_similarity as cs
    except Exception:
        return pd.DataFrame(index=range(len(texts)))

    model_name = "all-MiniLM-L6-v2"
    try:
        model = SentenceTransformer(model_name)
    except Exception:
        return pd.DataFrame(index=range(len(texts)))

    docs      = texts.fillna("").astype(str).tolist()
    themes    = list(seeds.keys())
    seed_docs = [" ".join(v) for v in seeds.values()]

    E_docs  = model.encode(docs,  show_progress_bar=False, normalize_embeddings=True)
    E_seeds = model.encode(seed_docs, show_progress_bar=False, normalize_embeddings=True)
    sims = cs(E_docs, E_seeds)
    return pd.DataFrame(sims, columns=[f"sim_embed__{t}" for t in themes])

embed_scores = embedding_similarity_scores(df_tagged["text_raw"], SEED_THEMES)

# -----------------------------
# Merge results
# -----------------------------
out = pd.concat([df_tagged.reset_index(drop=True), tfidf_scores, embed_scores], axis=1)

# -----------------------------
# Build audits (display only)
# -----------------------------
inv_col = COL_INVESTOR or "Investor"
audit_cols = [inv_col] + ([COL_ISSUER] if COL_ISSUER else []) + [
    "mentions_fossil_fuel_flag", "mentions_fossil_fuel_terms",
    "mentions_67pct_rule_flag",  "mentions_67pct_rule_terms"
]
audit_fossil = out[out["mentions_fossil_fuel_flag"]][audit_cols].sort_values(by=inv_col)
audit_67pct  = out[out["mentions_67pct_rule_flag"]][audit_cols].sort_values(by=inv_col)

# -----------------------------
# Display
# -----------------------------
def _h2(txt): 
    display(HTML(f"<h3 style='margin:8px 0 4px 0;font-family:Times New Roman'>{txt}</h3>"))

_h2("NLP Essentials — Summary")
summary = pd.DataFrame({
    "n_rows":[len(out)],
    "fossil_mentions":[int(out['mentions_fossil_fuel_flag'].sum())],
    "mentions_67pct":[int(out['mentions_67pct_rule_flag'].sum())]
})
display(summary)

_h2("Main Results (first rows)")
cols_order = [c for c in [COL_INVESTOR, COL_ISSUER, COL_TEXT, "text_norm"] if c in out.columns]
cols_order += [c for c in out.columns if c not in cols_order]  # keep derived metrics
display(out[cols_order].head(SHOW_N))

_h2("Audit — Investors mentioning fossil-fuel (first rows)")
display(audit_fossil.head(SHOW_N))

_h2("Audit — Investors mentioning '67%' (first rows)")
display(audit_67pct.head(SHOW_N))


,n_rows,fossil_mentions,mentions_67pct
0,4103,264,36


,investor_key,text,text_norm,text_raw,mentions_fossil_fuel_flag,mentions_fossil_fuel_terms,mentions_67pct_rule_flag,mentions_67pct_rule_terms,mentions_scope3_cat15_flag,mentions_scope3_cat15_terms,...,scopes_flag,scopes_terms,asset_classes_flag,asset_classes_terms,sim_tfidf__fossil_fuels,sim_tfidf__portfolio_coverage,sim_tfidf__net_zero_targets,sim_embed__fossil_fuels,sim_embed__portfolio_coverage,sim_embed__net_zero_targets
0,360 SECURITY TECHNOLOGY,No target https://www.360totalsecurity.com/en/about/ No target No sustainability policies of any kind found.,no target https //www 360totalsecurity com/en/about/ no target no sustainability policies of any kind found,No target https://www.360totalsecurity.com/en/about/ No target No sustainability policies of any kind found.,False,[],False,[],False,[],...,False,[],False,[],0.000000,0.008867,0.000000,0.238855,0.463088,0.404678
1,3I,"Other 2030 In corporate strategy https://www.3i.com/media/k5rhd4el/sustainability.pdf No target ""On 5 April 2023, we...",other 2030 in corporate strategy https //www 3i com/media/k5rhd4el/sustainability pdf no target on 5 april 2023 we w...,"Other 2030 In corporate strategy https://www.3i.com/media/k5rhd4el/sustainability.pdf No target ""On 5 April 2023, we...",False,[],False,[],True,[financed emissions],...,False,[],False,[],0.000780,0.063087,0.023699,0.265147,0.543503,0.491565
2,3M,"Carbon neutral(ity) 2050 In corporate strategy In 2015, we set a goal to have our Scope 1 and Scope 2 GHG emissions ...",carbon neutral ity 2050 in corporate strategy in 2015 we set a goal to have our scope 1 and scope 2 ghg emissions be...,"Carbon neutral(ity) 2050 In corporate strategy In 2015, we set a goal to have our Scope 1 and Scope 2 GHG emissions ...",False,[],False,[],False,[],...,False,[],False,[],0.000000,0.028493,0.013111,0.302739,0.509187,0.457448
3,77 BANK,"Climate neutral 2013 2030 In corporate strategy As for the 77 Bank Group’s CO2 emissions, achieve carbon neutrality ...",climate neutral 2013 2030 in corporate strategy as for the 77 bank group s co2 emissions achieve carbon neutrality b...,"Climate neutral 2013 2030 In corporate strategy As for the 77 Bank Group’s CO2 emissions, achieve carbon neutrality ...",False,[],False,[],False,[],...,False,[],False,[],0.005718,0.010671,0.026089,0.314826,0.505636,0.457193
4,AAREAL BANK,"No target 2023 In corporate strategy For the 2023 financial year, we have set ourselves the goal of ensuring carbon-...",no target 2023 in corporate strategy for the 2023 financial year we have set ourselves the goal of ensuring carbon n...,"No target 2023 In corporate strategy For the 2023 financial year, we have set ourselves the goal of ensuring carbon-...",False,[],False,[],False,[],...,False,[],False,[],0.000000,0.013894,0.015793,0.261827,0.439861,0.462313
5,ABA,No target No target,no target no target,No target No target,False,[],False,[],False,[],...,False,[],False,[],0.000000,0.000000,0.000000,0.062394,0.284540,0.439518
6,ABAKALIKI,No target No target,no target no target,No target No target,False,[],False,[],False,[],...,False,[],False,[],0.000000,0.000000,0.000000,0.062394,0.284540,0.439518
7,ABB,Net zero 100 2019 2050 In corporate strategy We’re partnering with our customers and suppliers to reduce their emiss...,net zero 100 2019 2050 in corporate strategy we re partnering with our customers and suppliers to reduce their emiss...,Net zero 100 2019 2050 In corporate strategy We’re partnering with our customers and suppliers to reduce their emiss...,False,[],False,[],False,[],...,False,[],False,[],0.000000,0.040152,0.021873,0.304772,0.443618,0.493672
8,ABBOTT LABORATORIES,Emissions reduction target 30 2018 2030 In corporate strategy • Reduce absolute Scope 1 and 2 carbon emissions by 30...,emissions reduction target 30 2018 2030 in corporate strategy reduce absolute scope 1 and 2 carbon emissions by 30% ...,Emissions reduction target 30 2018 2030 In corporate strategy • Reduce absolute Scope 1 and 2 carbon emissions by 30...,

,investor_key,mentions_fossil_fuel_flag,mentions_fossil_fuel_terms,mentions_67pct_rule_flag,mentions_67pct_rule_terms
14,ABRUZZO,True,[coal],False,[]
54,AES,True,"[coal, fossil fuel, fossil fuels, fossil-fuel]",False,[]
69,AGUASCALIENTES,True,[coal],False,[]
124,ALFA LAVAL AB,True,"[fossil fuel, fossil fuels, fossil-fuel]",False,[]
135,ALLIANT ENERGY,True,"[coal, fossil fuel, fossil-fuel]",False,[]
153,AMADEUS IT,True,"[fossil fuel, fossil fuels, fossil-fuel]",False,[]
158,AMAZONAS BRAZIL,True,[coal],False,[]
163,AMEREN,True,[coal],False,[]
211,ANTANANARIVO,True,[coal],False,[]
216,ANTWERPEN,True,[coal],False,[]


,investor_key,mentions_fossil_fuel_flag,mentions_fossil_fuel_terms,mentions_67pct_rule_flag,mentions_67pct_rule_terms
167,AMERICAN EXPRESS,False,[],True,"[two thirds, two-thirds]"
525,BLACKROCK,False,[],True,[67%]
546,BOOKING,False,[],True,[67%]
566,BRAZIL,False,[],True,[67 percent]
583,BROOKFIELD,False,[],True,"[two thirds, two-thirds]"
681,CBRE,False,[],True,[67%]
760,CHICAGO IL,False,[],True,[67%]
954,CP ALL,False,[],True,[67%]
970,CSL,False,[],True,[67%]
1040,DELOITTE,False,[],True,[67%]


In [4]:
# =============================================================================
# Summary stats for figure/table captions
# =============================================================================

# pick investor column (auto-detect from before)
inv_col = COL_INVESTOR or "Investor"

# unique investors in dataset
n_total = out[inv_col].nunique()

# investors with at least one fossil-fuel mention
n_fossil = out.loc[out["mentions_fossil_fuel_flag"], inv_col].nunique()

# investors with at least one 67% mention
n_67pct = out.loc[out["mentions_67pct_rule_flag"], inv_col].nunique()

pct_fossil = 100 * n_fossil / n_total if n_total else 0
pct_67pct  = 100 * n_67pct / n_total if n_total else 0

print(f"Total investors analysed: {n_total}")
print(f"Investors mentioning fossil fuels: {n_fossil} ({pct_fossil:.1f}%)")
print(f"Investors mentioning 67% rule:      {n_67pct} ({pct_67pct:.1f}%)")


Total investors analysed: 4103
Investors mentioning fossil fuels: 264 (6.4%)
Investors mentioning 67% rule:      36 (0.9%)
